# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring/ranking. I'm giving every page a score for how much it under-captures clicks vs. its position, then ranking pages by that score — not sorting into fixed categories.

In [8]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
import pandas as pd
from huggingface_hub import notebook_login, get_token # Import get_token and notebook_login

# Try to get the Hugging Face token using huggingface_hub's utility
hf_token = get_token()

# If no token is found, attempt to log in
if hf_token is None:
    print("HF_TOKEN not found. Attempting to log in to Hugging Face.")
    notebook_login() # This will prompt if not already logged in
    # Try to get the token again after login attempt
    hf_token = get_token()

if hf_token is None:
    # If still not found after attempting login, something is wrong
    print("Error: HF_TOKEN could not be found even after attempting login. Please check your Hugging Face token.")
    raise ValueError("HF_TOKEN not set, cannot proceed with data loading.")

# Ensure the token is set as an environment variable for httpfs to pick up
os.environ['HF_TOKEN'] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# Load just March 2026 as your working slice — avoids scanning all 78.8M rows
df_fact = con.sql(f"""
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(df_fact.shape)
df_fact.head()

df_fact["position_tier"] = pd.cut(df_fact["gsc_avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
df_fact["position_tier"].value_counts()

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 401)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target, CTR residual = actual CTR minus the median CTR for that page's position tier. Comes from observed data, not an outside rule. Big negative residual = under-capturing clicks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page = df_fact.groupby("content_hash_id").agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean")
).reset_index()
page["ctr"] = page["clicks"] / page["impressions"]
page["position_tier"] = pd.cut(page["avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")
page["ctr_residual"] = page["ctr"] - page["tier_median_ctr"]
page.head()


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the 50 worst-residual pages flagged, how many are real, high-volume candidates worth a reviewer's time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top50 = page.sort_values("ctr_residual").head(50)
top50[["content_hash_id","impressions","clicks","ctr","ctr_residual"]]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page, aggregated: total impressions, total clicks, average position, computed CTR.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(page.shape)
page.head()


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat CTR cutoff ignores that expected CTR depends on position — low CTR at position #2 means something different than at #15. The median-per-tier below proves CTR varies by position, so one fixed threshold can't work everywhere.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page.groupby("position_tier")["ctr"].median()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.